In [1]:
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity in the beginning
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

--------------------------------------------------------------------------
Ignoring value for oob_tcp_if_exclude on ccqlin065 (10.250.112.0/20: Did not find interface matching this subnet).
(You can safely ignore this message.)
--------------------------------------------------------------------------


Starting serial run at: 2026-04-29 13:23:48.818808


## From DFT to CoQui

<figure style="text-align: center;">
 <img src="../../images/coqui_workflow_wan90.png" alt="Workflow of CoQuí" width="60%">
</figure>

This notebook covers the entry point of a CoQuí many-body workflow: preparing reusable inputs from DFT results.

#### 🔹 Two necessary ingredients CoQuí needs

1. Crystal metadata (k-mesh, lattice, pseudopotentials, unit-cell information).

2. Single-particle Bloch orbitals used as the basis of the interacting many-electron problem. In practice, orbitals usually come from a mean-field method such as DFT, HF, or related single-particle approaches.

> Note: You can also start from a pre-optimized localized basis (for example, Gaussian-type orbitals), as long as the metadata and orbitals are provided in a supported format.

#### 🔹 Why MLWFs appear in this workflow?

Maximally localized Wannier functions (MLWFs) [[1](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.65.035109), [2](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.56.12847)] are a key bridge between different CoQuí components.

They are especially useful for:
- interpolation on dense k-meshes;
- defining physically motivated correlated subspaces for DMFT/EDMFT-style embedding.

#### 🔹 What this notebook covers

1. Convert QE outputs into CoQuí format with `pw2coqui.x`.

2. Declare the physical system through `coqui.Mf` (CoQuí's mean-field data container).

3. Construct MLWFs through CoQuí’s Wannier90 interface.

Reference material:
- QE tutorials: [official tutorials](https://www.quantum-espresso.org/tutorials/)

- Wannier90 references: [documentation](https://wannier90.readthedocs.io/en/latest/) and [tutorial slides](https://docs.epw-code.org/_downloads/f2f39ac545ca12dc2838f7359b3633a9/Mon.4.Pizzi.pdf)


First, let's copy the pre-computed Quantum ESPRESSO results and the necessary input files to the current directory.

In [2]:
%%bash
python copy_notebook_inputs.py

Copied: data/qe_inputs/svo/222/out -> out
Copied: data/qe_inputs/svo/222/svo.pw2coqui.in -> svo.pw2coqui.in
Copied: data/qe_inputs/svo/222/mlwf/svo.win -> mlwf/svo.win
Copied: data/qe_inputs/svo/222/mlwf_dp/svo.win -> mlwf_dp/svo.win
Notebook inputs are ready.


### Section 1: Quantum ESPRESSO converter

<figure style="text-align: center;">
 <img src="../../images/coqui_dft_converter.png" alt="Workflow of input preparation for CoQuí" width="60%">
</figure>

Electronic-structure packages such as QE, VASP, and PySCF write their results in different formats. CoQuí uses converter interfaces to translate those outputs into a unified input format for later many-body steps.

This section focuses on the Quantum ESPRESSO path and the QE converter [ `pw2coqui.x` ](https://github.com/AbInitioQHub/coqui/tree/main/qe_converter).

#### 🔹 Converter command

The QE converter is used from the command line:
>```bash
>pw2coqui.x -in {prefix}.pw2coqui.in
>```

Its minimal input file is:
>```text
>&input_pw2coqui
> prefix = "{prefix}"
> outdir = "{outdir}"
>/
>```
- `prefix` (required): QE prefix that identifies the dataset and output file names.

- `outdir` (required): QE output directory containing `{prefix}.save/`.

Running the converter generates:
- **`{prefix}.coqui.h5`**: standardized HDF5 metadata for CoQuí (k-point mesh, lattice vectors, pseudopotential information, etc.).

CoQuí also needs access to QE wavefunctions from the `nscf` step:
- **`{outdir}/{prefix}.save/`**: directory containing Kohn–Sham orbitals (`wfc*.hdf5`) used as the Bloch basis.

Together, these form the complete input for subsequent CoQuí steps.


> Important: Do not move or delete `{prefix}.save/` after conversion. Later steps require both `.coqui.h5` and `{prefix}.save/`.


#### 🔹 Hands-on — Converting SrVO$_3$ results to CoQui format

In [3]:
%%bash
# 1. Runing pw2coqui.x to convert the QE output to CoQui format
# 2. Inspect the content of the generated HDF5 file
pw2coqui.x -i svo.pw2coqui.in
h5ls ./out/svo.coqui.h5/Orbitals

--------------------------------------------------------------------------
Ignoring value for oob_tcp_if_exclude on ccqlin065 (10.250.112.0/20: Did not find interface matching this subnet).
(You can safely ignore this message.)
--------------------------------------------------------------------------



     Program PW2AIMBES v.7.4 starts on 29Apr2026 at 13:24: 6 
        Git branch: master
        Last git commit: a546703a761ccaf2e961f281526947e5269d4e69-dirty
        Last git commit date: Mon Oct 21 12:38:43 2024 +0000
        Last git commit subject: Merge branch 'master-qe-v7.4' into 'master'

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Parallel version (MPI & OpenMP), running on      32 processor cores
     Number of MPI processes:                 1
     Threads/MPI process:                    32

     MPI processes dist

### Section 2: Declaring a simulated physical system

<figure style="text-align: center;">
 <img src="../../images/coqui_workflow_mf.png" alt="Workflow of CoQuí's Mf declaration step" width="60%">
 <figcaption><em>Figure&nbsp;2:</em> The mean-field declaration step in the CoQuí workflow.</figcaption>
</figure>

A universal starting point for CoQui simulations is declaring the simulated physical system. In CoQui, a simulated system is defined by 

- metadata (lattice, k-mesh, pseudopotentials, etc.) 

- the single-particle orbitals $\phi^{\mathbf{k}}_{i}(\mathbf{r})$, 

which together determine the non-interacting Hamiltonian: 
$$
(H_{0})^{\textbf{k}}_{ij} = \int d\textbf{r} \, \phi^{\textbf{k}*}_{i}(\textbf{r}) \Big [ \frac{\nabla^{2}}{2} + V_{\mathrm{ext}}(\textbf{r}) \Big ] \phi^{\textbf{k}}_{j}(\textbf{r}),
$$

#### 🔹 `coqui.Mf` container

In CoQuí, the declaration of the simulated system is handled by the read-only `coqui.Mf` container, constructed through:
>```python
>coqui.make_mf(mpi: coqui.MpiHandler, params: dict, mf_type: str) -> coqui.Mf
>```

Function inputs:
- `mpi` (required): `MpiHandler` instance that carries the MPI context.

- `params` (required): dictionary containing input parameters. Important keys include:
    - `prefix` (required): QE prefix used in `scf/nscf`.

    - `outdir` (required): QE outdir containing `{prefix}.save/`.

    - `nbnd` (optional): number of imported bands. If omitted, all available bands are imported.

- `mf_type` (required): DFT-backend, currently `qe` for Quantum ESPRESSO or `pyscf` for PySCF.

The data structure of the returned `coqui.Mf` object is independent of the underlying DFT backend (QE, PySCF, etc.). `coqui.Mf` therefore serves as a universal starting point for all CoQuí calculations, providing a consistent interface across different DFT backends.

> Tip: Manipulate `nbnd` to control the number of KS orbitals for basis convergence tests and faster experimental runs. 

#### 🔹 Example

The following example constructs a `Mf` object for SrVO$_3$ from a pre-computed DFT calculation using Quantum ESPRESSO, converted to CoQui format in the previous section.

In [ ]:
# Mf for the target system
params = {
 "prefix": "svo",     # QE prefix (matches {prefix}.save)
 "outdir": "out",     # QE outdir containing {prefix}.save/
 "nbnd": 40           # number of bands imported from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

#### 🔹 Hands-on — `Mf` container for SrVO$_3$ from Quantum ESPRESSO

Create an `Mf` object for SrVO$_3$ using the example above, then complete the checks below.

1. Describe the system from the log output: number of KS orbitals, k-points, spins, and related metadata.

2. Cross-check with QE [input](../../data/qe_inputs/svo/222/out/svo.nscf.in)/[output](../../data/qe_inputs/svo/222/out/svo.nscf.out) and verify that the `Mf` metadata matches the QE setup.

3. Experiment with `"nbnd"`:
 - Remove `"nbnd"` and rebuild `Mf` to include all KS orbitals. What is the total count?
 
 - Reintroduce `"nbnd"` with different values and verify the reported number of included orbitals.

In [4]:
# Mf for the target system
params = {
  "prefix": "svo",                        # QE prefix (matches {prefix}.save)
  "outdir": "out", # QE outdir containing {prefix}.save/
  "nbnd": 40                              # number of bands read from QE outputs
}
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# remove "nbnd"
del params["nbnd"]
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

# set "nbnd" = 5
params["nbnd"] = 5
mf = coqui.make_mf(coqui_mpi, params=params, mf_type="qe")

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 100
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

  Quantum ESPRESSO reader
  -----------------------
  Numbe

### Section 3: MLWFs from CoQuí

<figure style="text-align: center;">
 <img src="../../images/coqui_workflow_wan90.png" alt="Workflow of CoQuí's Wannier90 interface" width="60%">
 <figcaption><em>Figure&nbsp;2:</em> MLWF construction step in the CoQuí workflow.</figcaption>
</figure>

Maximally localized Wannier functions (MLWFs) [[1](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.65.035109), [2](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.56.12847)] are a key bridge between different CoQuí components.

They are especially useful for:
- interpolation on dense k-meshes;

- defining physically motivated correlated subspaces for DMFT/EDMFT-style embedding.

This section focuses on calling Wannier90 from inside CoQuí so that Wannierization becomes part of the same Python workflow.

#### 🔹 Wannier90 interface

CoQuí exposes the Wannier90 interface through:

> ```python
> coqui.wannier90(mf: coqui.Mf, params: dict)
> ```

Function inputs:
- `mf` (required): `Mf` instance for the target system.

- `params` (required): dictionary containing Wannier90 interface parameters. Important keys include:
    - `prefix` (required): Wannier90 seedname used to locate `{prefix}.win` and write outputs

    - `h5_filename` (optional): Name of the HDF5 file to store the MLWF data.

External parameter file:
- `{prefix}.win` (required): Standard Wannier90 input file present in the working directory. Specifies Wannierization settings (projections, band window, number of Wannier functions, etc.). Atomic positions, lattice vectors, and k-points are filled in automatically by CoQuí and do not need to be included.


    >```text
    ># Example: svo.win
    >num_wann = 3
    >exclude_bands : 1-20,24-40
    >num_bands = 3
    >
    >begin projections
    >V:dxz,dyz,dxy:x=1,0,0
    >end projections
    >```

Output:
- `{h5_filename}` (default: `{prefix}.mlwf.h5`): HDF5 file storing the Wannierization results, including MLWF centers, and the corresponding projectors going from the Bloch basis to the Wannier basis. This file is the primary input for subsequent CoQuí steps that require MLWFs.


#### 🔹 Example

In the follwoing example, we first build the `Mf` object for SrVO$_3$ and then call `coqui.wannier90` with a prepared `svo.win` file to construct MLWFs.

In [5]:
# Step 1: Build mean-field object from QE outputs
mf_params = {
    "prefix": "svo",
    "outdir": "out",
    "nbnd": 40,
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {
    "prefix": "mlwf/svo",
    "h5_filename": "mlwf/svo.mlwf.h5"
}
coqui.wannier90(mf=mf, params=w90_params)

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

*************************************************
       Running Wannier90 in library-mode         
*************************************************

Note: no parallel distribution provided (option distk missing)
Note: all k-points handled by MPI rank 0

 *---------------------------------- K-MESH ----------------------------------*
 +----------------------------------------------------------------------------+
 |                    Distance to Nearest-Neighbour Shells                    |
 |                    ------------

Questions:
1. How many MLWFs are generated?

2. What are their spreads and centers?

#### 🔹 Hands-on — MLWFs with entanglement

Continue the Section 3 hands-on by modifying `svo.win` to build MLWFs for both V $d$ and O $p$ states using a larger disentanglement window.

Steps:
1. Edit `svo.win` to include O $p$ bands and the full V $d$ shell.

2. Run `coqui.wannier90` again to generate updated MLWFs.

3. Compare against the small-window result from Hands-on 2a.

Checklist:
- MLWF spreads should become noticeably smaller than in the small-window case.

- Each oxygen atom should have three localized $p$-like orbitals.

Reference: oxygen positions are listed in `data/qe_inputs/svo/222/out/svo.nscf.out`.


In [6]:
# Step 1: Build mean-field object from existing QE outputs
mf_params = {
    "prefix": "svo",
    "outdir": "out",
    "nbnd": 40,
}
mf = coqui.make_mf(coqui_mpi, params=mf_params, mf_type="qe")

# Step 2: Run Wannier90 through CoQuí
w90_params = {
    "prefix": "mlwf_dp/svo",
    "h5_filename": "mlwf_dp/svo.mlwf.h5",
}
coqui.wannier90(mf=mf, params=w90_params)

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 40
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 41.0
  Electron density energy cutoff = 360.000 a.u. | FFT mesh = (45,45,45)
  Wavefunction energy cutoff     = 51.704 a.u. | FFT mesh = (23,23,23), Number of PWs = 6859

*************************************************
       Running Wannier90 in library-mode         
*************************************************

Note: no parallel distribution provided (option distk missing)
Note: all k-points handled by MPI rank 0

 *---------------------------------- K-MESH ----------------------------------*
 +----------------------------------------------------------------------------+
 |                    Distance to Nearest-Neighbour Shells                    |
 |                    ------------

12

     20     0.000E+00     0.0000000726        9.8538521157     107.76  <-- CONV
        O_D=      0.0000000 O_OD=      0.1708332 O_TOT=      9.8538521 <-- SPRD
 Delta: O_D= -0.2292329E-17 O_OD= -0.6938894E-15 O_TOT=  0.0000000E+00 <-- DLTA
 ------------------------------------------------------------------------------
 Cycle:     30
  WF centre and spread    1  ( -0.000000, -0.000000,  0.000000 )     0.65456162
  WF centre and spread    2  ( -0.000000,  0.000000, -0.000000 )     0.65456162
  WF centre and spread    3  ( -0.000000,  0.000000, -0.000000 )     0.65456162
  WF centre and spread    4  (  0.000000,  0.000000, -0.000000 )     0.56243732
  WF centre and spread    5  (  0.000000, -0.000000,  0.000000 )     0.56242661
  WF centre and spread    6  (  1.920510, -0.000000, -0.000000 )     0.80562725
  WF centre and spread    7  (  1.920510, -0.000000, -0.000000 )     0.64382839
  WF centre and spread    8  (  1.920510,  0.000000, -0.000000 )     0.80562725
  WF centre and sprea

Add a section to do band_interpolation with MLWFs to provide some visualization before students get bored. 